# 🚀 SAM-Road (2024) Zero-Shot Domain Transfer & Inference on PCB Imagery

This notebook sets up Meta's **SAM-Road (2024)** architecture on Kaggle Dual Tesla T4 GPUs, fetches pretrained HuggingFace weights (`congrui/sam_road`), pre-processes PCB board photos to 512x512, and runs end-to-end graph extraction.

### Settings Required:
- **Accelerator**: GPU T4 x2 (or P100)
- **Internet**: **ON**


In [ ]:
# 1. Clone SAM-Road Repository & Install Dependencies
import os
!git clone https://github.com/htcr/sam_road.git
%cd sam_road
!pip install pytorch-lightning wandb opencv-python scipy shapely rtree huggingface_hub -q


In [ ]:
# 2. Download SAM Base ViT-B Backbone Checkpoint
!mkdir -p sam_ckpts
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O sam_ckpts/sam_vit_b_01ec64.pth
print('✅ SAM ViT-B Base Checkpoint Downloaded!')


In [ ]:
# 3. Download Fine-Tuned SAM-Road Checkpoint from HuggingFace Hub
from huggingface_hub import hf_hub_download
try:
    ckpt_path = hf_hub_download(repo_id='congrui/sam_road', filename='cityscale_vitb_512_e10.ckpt')
    print('✅ Downloaded HuggingFace SAM-Road Checkpoint to:', ckpt_path)
except Exception as e:
    print('Error downloading checkpoint:', e)


In [ ]:
# 4. Prepare PCB Image for SAM-Road (512x512 RGB)
import cv2, glob, numpy as np
from pathlib import Path

# Search for PCB photos or use demo photo
pcb_imgs = glob.glob('/kaggle/input/**/*.jpg', recursive=True) + glob.glob('/kaggle/working/**/*.jpg', recursive=True)
sample_pcb = pcb_imgs[0] if pcb_imgs else 'sample_pcb.jpg'

img = cv2.imread(sample_pcb) if os.path.exists(sample_pcb) else (255 * np.ones((512, 512, 3), dtype=np.uint8))
img_512 = cv2.resize(img, (512, 512))
cv2.imwrite('sample_pcb_512.png', img_512)
print('✅ Resized PCB Image Saved to sample_pcb_512.png')


In [ ]:
# 5. Run SAM-Road Zero-Shot Inference
!python inferencer.py --config=config/toponet_vitb_512_cityscale.yaml --checkpoint={ckpt_path}


In [ ]:
# 6. Render & Display Visual Graph Output
from PIL import Image
import glob
outputs = glob.glob('/kaggle/working/**/*.png', recursive=True)
for out in outputs[:3]:
    print('Displaying:', out)
    display(Image.open(out))
